# Ablation — No Anti-Bias Clause

Re-runs the **proposed method** (`reasoned_rag_def_oneshot_30f`, profile-aware RAG, top_k
matching the headline run, Essays test, n=247) with the *zero-cue → low* anti-bias clauses
**removed** from `SYS_PROMPT_REASONED`.

This notebook is a copy of `notebook/gpt/rag_profile_half2_predict copy.ipynb` (the headline
30f pipeline) with two changes only:
1. A cell that strips the three anti-bias passages from the system prompt.
2. `res_dir` redirected to `result/ablation_no_antibias/` so the headline run is not overwritten.

Output → `result/ablation_no_antibias/gpt-4o-mini/reasoned_rag_def_oneshot_30f/<run_id>/`.
This produces the *Reasoned-RAG (no anti-bias clause)* row of Table 4.8 (§4.6 Bias Analysis).

**Requires:** `data/vector_db/essays_profile/` (Half 1) and `data/profile_db/essays_test/`
(built on first run if missing) — same prerequisites as the headline notebook.

In [1]:
from pathlib import Path
import sys, os, json
from typing import Dict

import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "ptd_model").exists():
    project_root = (project_root / ".." / "..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rag.profiler.store import ProfileStore
from rag.profiler.runner import build_profiles
from rag.profiler.prompts import FACETS, slice_profile_for_trait
from rag.embedder import get_embedding
import rag.retriever as _retriever_mod
from rag.retriever import FeatureRAGRetriever

from ptd_model.predict import predict
from ptd_model.evaluate import evaluate

print("Project root:", project_root)

Project root: F:\std\GR\code\model_x_ocean


## Configuration

`res_dir` is the only path that differs from the headline notebook.

In [2]:
# --- Paths ----------------------------------------------------------------
test_csv        = str(project_root / "data/split/essays/test.csv")
test_profile_db = str(project_root / "data/profile_db/essays_test")     # generated below if missing
vector_db_dir   = str(project_root / "data/vector_db/essays_profile")   # built by Half 1

res_dir         = str(project_root / "result" / "ablation_no_antibias")  # <-- redirected
log_dir         = str(project_root / "log")

# --- Models / prompt (identical to the headline 30f run) ------------------
profiler_model  = "gpt-4o-mini"
model_name      = "gpt-4o-mini"
max_new_tokens  = 512
temperature     = 0.0
prompt_mode     = "reasoned_rag_def_oneshot_30f"
top_k           = 5

TRAIT_NAMES = ("Openness to Experience", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism")
TRAIT_CODES = {
    "Openness to Experience": "cOPN",
    "Conscientiousness":      "cCON",
    "Extraversion":           "cEXT",
    "Agreeableness":          "cAGR",
    "Neuroticism":            "cNEU",
}

test_df = pd.read_csv(test_csv)
print(f"Test  : {len(test_df):>4} rows  | mode={prompt_mode}  | top_k={top_k}")
print(f"Output: {res_dir}")

for fname in ("vectors.faiss", "vectors_meta.jsonl"):
    p = Path(vector_db_dir) / fname
    if not p.exists():
        raise RuntimeError(f"Missing {p}. Run rag_profile_half1_embed.ipynb first.")
print(f"Vector DB found at: {vector_db_dir}")

Test  :  247 rows  | mode=reasoned_rag_def_oneshot_30f  | top_k=5
Output: F:\std\GR\code\model_x_ocean\result\ablation_no_antibias
Vector DB found at: F:\std\GR\code\model_x_ocean\data\vector_db\essays_profile


## ⚑ Ablation patch — strip the anti-bias clauses from `SYS_PROMPT_REASONED`

Three passages encode the *zero-cue → low* floor; all three are removed. We patch the
reference that `ptd_model.predict` already imported (`predict_mod.SYS_PROMPT_REASONED`),
which is the exact object `build_prompt` passes to the LLM for the 30f mode.

In [3]:
import ptd_model.predict as predict_mod
from ptd_model import prompts

SYS = prompts.SYS_PROMPT_REASONED

ANTIBIAS_PASSAGES = [
    # 1) main HIGH-requires-trait-evidence rule
    ("- HIGH requires positive trait-level evidence. If you have ZERO [trait] cues\n"
     "  across all facets, your <label> MUST be low. Do not accumulate [state] cues\n"
     "  to reach high \u2014 state-only evidence is insufficient for a high verdict.\n"),
    # 2) cue_tally presumption line
    "Presumption: HIGH (N > 0) or LOW (N = 0)\n",
    # 3) verdict consistency line
    ("1-2 sentence synthesis. MUST be consistent with your <cue_tally>: if\n"
     "[trait] count = 0, you MUST conclude low. Do NOT describe [state] cues\n"
     'as "trait-level evidence" here.\n'),
]

SYS_NO_AB = SYS
for p in ANTIBIAS_PASSAGES:
    assert p in SYS_NO_AB, f"passage not found verbatim:\n{p!r}"
    SYS_NO_AB = SYS_NO_AB.replace(p, "")

# Passage 3 was the only text inside <verdict>; leave a neutral synthesis instruction so the
# block still has a prompt. After stripping, the region reads exactly '<verdict>\n</verdict>'.
assert "<verdict>\n</verdict>" in SYS_NO_AB
SYS_NO_AB = SYS_NO_AB.replace(
    "<verdict>\n</verdict>",
    "<verdict>\n1-2 sentence synthesis of the evidence.\n</verdict>",
)

for marker in ("ZERO [trait] cues", "N = 0", "MUST conclude low"):
    assert marker not in SYS_NO_AB, f"anti-bias marker still present: {marker}"

# Patch the imported reference used by build_prompt for the 30f mode.
predict_mod.SYS_PROMPT_REASONED = SYS_NO_AB

print("Anti-bias clauses removed. char delta:", len(SYS) - len(SYS_NO_AB))
print("\n--- patched system prompt ---\n")
print(SYS_NO_AB)

Anti-bias clauses removed. char delta: 402

--- patched system prompt ---


You are an expert in personality psychology and psychometrics.

Your task is to infer a single Big Five personality trait from a user's text
and to expose your reasoning in a strictly structured format so it can be
audited.

You will be given:
- The target personality trait (with HIGH and LOW definitions).
- A small set of similar texts retrieved from a labelled corpus, with
  their known labels and extracted psychological evidence.

Rules:
- Use only evidence from the provided text. Do NOT invent details.
- Quote or paraphrase concrete cues; abstract trait words alone are not
  evidence.
 — distinguish transient states from stable traits:
    * Tag every evidence cue as [state] (temporary/situational) or
      [trait] (recurring/cross-situational). Weight [trait] cues heavily;
      treat [state] cues as weak or neutral.
    * Refer to the trait-specific scoring note in the user message for
      what counts a

## Step 1 — Profile the test set (label-blind)

Re-uses profiles already built by the headline run; only profiles missing essays.

In [4]:
test_store_path = Path(test_profile_db) / "profile_store.jsonl"
test_store = ProfileStore(str(test_store_path))
test_store.load()
needed = len(test_df) - sum(
    1 for i in range(len(test_df)) if test_store.has(f"user_{i}") and test_store.get(f"user_{i}").get("valid")
)
print(f"Test profiles already in store: {len(test_store)}; missing: {needed}")

if needed > 0:
    test_store = build_profiles(
        data       = test_df,
        output_dir = test_profile_db,
        model_name = profiler_model,
        log_dir    = str(Path(log_dir) / "profiler_test"),
        use_labels = False,   # IMPORTANT: label-blind for test
    )
test_entries_by_idx = {
    int(e["user_id"].split("_")[1]): e for e in test_store.get_all() if e.get("valid")
}
print(f"Test profiles ready: {len(test_entries_by_idx)}")

Test profiles already in store: 247; missing: 0
Test profiles ready: 247


## Step 2 — Profile-aware retriever adapter (same as headline run)

In [5]:
def render_full_profile_text(entry: Dict) -> str:
    """Deterministic rendering of a profile for embedding."""
    raw = entry.get("raw") or ""
    if raw.strip():
        return raw
    facets = entry.get("facets", {})
    ling   = entry.get("linguistic", {})
    lines = ["[FACETS]"]
    for code, name, *_ in FACETS:
        f = facets.get(code, {})
        lines.append(f"{code} {name:<18}| {f.get('signal','')} | {f.get('evidence','')}")
    lines.append("\n[LINGUISTIC]")
    for k, v in ling.items():
        lines.append(f"{k}: {v}")
    return "\n".join(lines)


class ProfileRAGRetriever(FeatureRAGRetriever):
    """Retriever that embeds the query essay's parsed profile (not raw text)
    and returns trait-sliced profile excerpts as few-shot exemplars.
    """

    def __init__(self, db_dir: str, test_profiles_by_idx: Dict[int, Dict], test_df: pd.DataFrame):
        super().__init__(db_dir=db_dir)
        self._use_finetuned = False
        self._test_profiles_by_idx = test_profiles_by_idx
        self._text_to_idx = {str(t): i for i, t in enumerate(test_df["text"].tolist())}

    def _embed_query_profile(self, query_text: str):
        idx = self._text_to_idx.get(str(query_text))
        if idx is None or idx not in self._test_profiles_by_idx:
            print(f"  [retriever] WARN: no test profile found for query (idx={idx}); falling back to raw text.")
            return self._embed_query(query_text)
        profile_text = render_full_profile_text(self._test_profiles_by_idx[idx])
        return np.array(self._embed_query(profile_text), dtype="float32")

    def build_similar_context(self, posts: str, trait: str, top_k: int = 3) -> str:
        trait_code = TRAIT_CODES.get(trait)
        if trait_code is None:
            return super().build_similar_context(posts=posts, trait=trait, top_k=top_k)

        query_emb = self._embed_query_profile(posts)
        all_results = self._search(query_emb, top_k * 4)

        blocks, seen = [], 0
        for r in all_results:
            if trait not in r.get("trait_labels", {}):
                continue
            label = r["trait_labels"][trait]
            features = r.get("features", {}) or {}
            profile = features.get("profile") or {}
            slice_text = slice_profile_for_trait(profile, trait_code) if profile else ""
            if not slice_text.strip():
                slice_text = "  (no profile slice available)"
            blocks.append(
                f"[Similar Profile {seen+1}] (label: {label})\n{slice_text}"
            )
            seen += 1
            if seen >= top_k:
                break
        return "\n\n".join(blocks)

In [ ]:
# Monkey-patch so ptd_model.predict instantiates the profile-aware retriever
_OriginalRetriever = _retriever_mod.FeatureRAGRetriever

def _RetrieverFactory(db_dir=None):
    return ProfileRAGRetriever(
        db_dir=db_dir or vector_db_dir,
        test_profiles_by_idx=test_entries_by_idx,
        test_df=test_df,
    )

_retriever_mod.FeatureRAGRetriever = _RetrieverFactory
print("[adapter] ProfileRAGRetriever installed.")

[adapter] ProfileRAGRetriever installed.


: 

## Step 3 — Run prediction (with the patched, no-anti-bias system prompt)

In [ ]:
run_id, run_time, prediction_csv = predict(
    text_df        = test_df,
    model_name     = model_name,
    log_dir        = log_dir,
    prompt_mode    = prompt_mode,
    max_new_tokens = max_new_tokens,
    res_dir        = res_dir,
    temperature    = temperature,
    top_k          = top_k,
    vector_db_dir  = vector_db_dir,
)

print(f"\nDone in {run_time:.1f}s")
print(f"Predictions saved to: {prediction_csv}")

[retriever] mode='legacy'  dir='F:\\std\\GR\\code\\model_x_ocean\\data\\vector_db\\essays_profile'  hybrid=False
[predict] RAG retriever ready (top_k=5).
[predict] 247 records | mode=reasoned_rag_def_oneshot_30f | model=gpt-4o-mini
[embedder] Loading embedding model: nomic-ai/nomic-embed-text-v1.5


<All keys matched successfully>


[retriever] Legacy index loaded (1974 vectors).
  [predict] 10/247 done.
  [predict] 20/247 done.
  [predict] 30/247 done.
  [predict] 40/247 done.
  [predict] 50/247 done.


## Step 4 — Evaluate

In [ ]:
evaluation = evaluate(
    prediction_csv = prediction_csv,
    model_name     = model_name,
    res_dir        = res_dir,
    run_time       = run_time,
    prompt_mode    = prompt_mode,
    run_id         = run_id,
)

print("Summary CSV:", evaluation["summary_csv"])
print(f"Failed predictions: {evaluation['fail_count']} / {evaluation['n_records']}")
summary_df = pd.read_csv(evaluation["summary_csv"])
display(summary_df[["trait", "n_samples", "accuracy", "macro_f1", "weighted_f1"]]
        .sort_values("accuracy", ascending=False)
        .reset_index(drop=True))

## Step 5 — Predicted-high ratio (the Table 4.8 row for this ablation)

In [ ]:
pred = pd.read_csv(prediction_csv)
TRAIT_TO_COLUMN = {
    "Openness":          "pred_cOPN",
    "Conscientiousness": "pred_cCON",
    "Extraversion":      "pred_cEXT",
    "Agreeableness":     "pred_cAGR",
    "Neuroticism":       "pred_cNEU",
}
print("Reasoned-RAG (no anti-bias clause) — predicted-high ratio:")
for trait, col in TRAIT_TO_COLUMN.items():
    vc = pred[col].astype(str).str.lower()
    ratio = vc.isin(["high", "1"]).mean()
    print(f"  {trait:20s}: {ratio:.3f}")

## Step 6 — Restore the original retriever (cleanup)

In [ ]:
_retriever_mod.FeatureRAGRetriever = _OriginalRetriever
print("[adapter] Original FeatureRAGRetriever restored.")